# Vision base model baseline: visual 2-step transformation diagnoser

Baseline for the vision path. The model is shown a coordinate-grid image (the pre-image polygon plus the student's proposed final image) and the two transformation steps in text, and it must return the misconception diagnosis as JSON. This measures a vision base model before any fine-tuning.

**Base model:** `unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit` (Qwen3-VL, 4B, fits a free T4 in 4-bit).

**Runtime:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all.

**Why visual input:** reading positions off a grid is where frontier models are documented to be weakest, so it is the cleanest place for a fine-tuned specialist to beat a prompted frontier model. Ground truth stays exact because the diagram is generated from known coordinates.

This supersedes the text-only notebook (`01_base_model_inference.ipynb`) for the input format; the transform math and taxonomy are reused here.

In [ ]:
# Confirm a GPU is attached (expect a T4 on free Colab).
!nvidia-smi

In [ ]:
# Install Unsloth (includes the vision fine-tuning stack). Output is visible so errors show.
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo
# If pip upgrades torch, do Runtime > Restart session once, then run the cells below (skip this one).

In [ ]:
from unsloth import FastVisionModel
import torch

MODEL_NAME = "unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit"

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
FastVisionModel.for_inference(model)  # enable Unsloth's faster inference path
print("loaded:", MODEL_NAME)

In [ ]:
# v0 misconception taxonomy (draft, to validate) and the instruction the model receives with the image.
MISCONCEPTIONS = {
    "order_swapped": "Applied the two transformations in the wrong order.",
    "center_not_updated": "Did not update the center or reference frame for the second transformation.",
    "wrong_direction": "Rotated or translated in the wrong direction or sign.",
    "reflected_wrong_axis": "Reflected over the wrong axis.",
    "only_first_applied": "Applied only the first transformation and skipped the second.",
    "arithmetic_slip": "Correct method but a small coordinate arithmetic error.",
    "none": "The student's answer is correct.",
}

INSTRUCTION = (
    "You are shown a coordinate grid. The solid polygon labeled P is the pre-image. "
    "The dashed polygon labeled S is a student's proposed final image after applying two "
    "transformations, in order, to P.\n"
    "Diagnose the student's error. Return a SINGLE valid JSON object and nothing else: "
    "no prose, no markdown, no code fences.\n"
    "Required fields:\n"
    '  "target_quantity": what is being asked for (the vertices of the final image).\n'
    '  "correct_answer": the correct final vertices, which you compute from P and the steps.\n'
    f'  "misconception_label": exactly one of {sorted(MISCONCEPTIONS)}.\n'
    '  "evidence_span": the student vertices (S) that show the error.\n'
    '  "hint": a Socratic nudge that does NOT state the correct final answer.\n'
    'Use "none" only if S is correct. The hint must never reveal correct_answer.'
)
print(INSTRUCTION)

In [ ]:
# Exact integer rigid motions, so every rendered problem has programmatic ground truth.
def translate(pts, dx, dy):
    return [(x + dx, y + dy) for (x, y) in pts]

def rotate(pts, center, deg):
    cx, cy = center
    out = []
    for (x, y) in pts:
        x0, y0 = x - cx, y - cy
        if deg == 90:
            xr, yr = -y0, x0
        elif deg == 180:
            xr, yr = -x0, -y0
        elif deg == 270:
            xr, yr = y0, -x0
        else:
            raise ValueError(f"unsupported angle: {deg}")
        out.append((xr + cx, yr + cy))
    return out

def reflect(pts, axis):
    fns = {
        "x": lambda x, y: (x, -y),
        "y": lambda x, y: (-x, y),
        "y=x": lambda x, y: (y, x),
        "y=-x": lambda x, y: (-y, -x),
    }
    f = fns[axis]
    return [f(x, y) for (x, y) in pts]

def fmt(pts):
    return ", ".join(f"({x}, {y})" for (x, y) in pts)

In [ ]:
import io
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

def render_problem(pre, student, lim=8):
    """Draw the pre-image P (solid) and the student's proposed final image S (dashed) on a grid.
    Positions must be read from the grid; the transformations are given separately as text."""
    fig, ax = plt.subplots(figsize=(4.2, 4.2), dpi=120)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.grid(True, linewidth=0.5, alpha=0.4)
    ax.axhline(0, color="black", linewidth=1)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xticks(range(-lim, lim + 1, 2))
    ax.set_yticks(range(-lim, lim + 1, 2))

    def draw(pts, color, style, marker, label):
        xs = [p[0] for p in pts] + [pts[0][0]]
        ys = [p[1] for p in pts] + [pts[0][1]]
        ax.plot(xs, ys, color=color, linewidth=2, linestyle=style, marker=marker, label=label)

    draw(pre, "#1f4e79", "-", "o", "P (pre-image)")
    draw(student, "#b23a2e", "--", "s", "S (student's answer)")
    ax.legend(loc="upper left", fontsize=8, framealpha=0.9)

    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).convert("RGB")

# One sample problem with an injected misconception.
PRE = [(1, 1), (4, 1), (1, 3)]
STEPS_TEXT = "Step 1: translate by (2, -1). Step 2: rotate 90 degrees counterclockwise about the origin."
step1 = lambda p: translate(p, 2, -1)
step2 = lambda p: rotate(p, (0, 0), 90)

correct = step2(step1(PRE))            # translate THEN rotate
student = step1(step2(PRE))            # rotate THEN translate -> order_swapped
true_label = "order_swapped"

img = render_problem(PRE, student)
print("correct:", fmt(correct), "| student S:", fmt(student), "| injected label:", true_label)
img

In [ ]:
import json, re

def diagnose(image, steps_text):
    user_text = INSTRUCTION + "\n\n" + steps_text
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": user_text},
    ]}]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    gen = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

def check(text, correct, true_label):
    result = {"valid_json": False, "json_only": False, "label_valid": False,
              "label_correct": None, "no_leak": None}
    obj = None
    try:
        obj = json.loads(text)
        result["valid_json"] = True
        result["json_only"] = True
    except Exception:
        m = re.search(r"\{.*\}", text, re.S)
        if m:
            try:
                obj = json.loads(m.group(0))
                result["valid_json"] = True
            except Exception:
                obj = None
    if isinstance(obj, dict):
        label = obj.get("misconception_label")
        result["label_valid"] = label in MISCONCEPTIONS
        result["label_correct"] = (label == true_label)
        hint = str(obj.get("hint", ""))
        result["no_leak"] = all(
            f"({x}, {y})" not in hint and f"({x},{y})" not in hint
            for (x, y) in correct
        )
    return result

In [ ]:
# Run the vision base model on the rendered problem. This is the baseline the fine-tune has to beat.
raw = diagnose(img, STEPS_TEXT)
print(raw)
print("\ncheck:", check(raw, correct, true_label))

## Reading the baseline

Look at whether the model:
- returned one clean JSON object (`json_only`),
- read the grid well enough to compute the right `correct_answer` (the perception test),
- named the injected misconception (`label_correct`),
- kept the answer out of the hint (`no_leak`).

Expect the base model to struggle most with reading positions off the grid, which is the failure this project targets. That is the gap the fine-tune closes, and the axis where a specialist can beat a prompted frontier VLM.

## Next steps
1. Data generator: render a few thousand problems (varied polygons, all four composition types, varied translations, centers, axes), each an image plus exact ground-truth answer and injected misconception label. Save as an image dataset with the JSON target.
2. Eval harness: the JSON, label, and no-leak checks plus solution-correctness (compare `correct_answer` to ground truth), run base vs tuned vs a prompted frontier VLM on a held-out split, and report extraction/solve accuracy separately from label accuracy.
3. Vision QLoRA fine-tune with `FastVisionModel.get_peft_model` and `UnslothVisionDataCollator`, then re-run the eval.